# Matrix elements from the summed GEVP (sGEVP), forward

The effective matrix element of Khan et al. (arXiv:2107.08960, eq. A7), with the summed GEVP of Bulava, Donnellan, Sommer (arXiv:1108.3774):

- $K_{ab}(t) = \sum_{\tau=1}^{t-1} \left[\langle C_{ab}(t)\, O(\tau)\rangle - \langle C_{ab}(t)\rangle\langle O(\tau)\rangle\right]$: the 3pt of method 1, summed over the operator time;
- GEVP $C(t)\,u = \lambda\, C(t_0)\,u$, solved again at every $t$ and for every jackknife sample, $u^\dagger C(t_0)\, u = 1$;
- $R(t) = u^\dagger K(t)\, u / \lambda(t, t_0) - u^\dagger K(t_0)\, u$,  $M_{\rm eff}(t) = R(t+1) - R(t)$;
- fit $M_{\rm eff}(t) = A + B\, t\, e^{-\Delta E\, t}$: $A$ is the matrix element.

The derivation is in `~/Desktop/785cfg_report/meff_sgevp_derivation.pdf`. Same data sets, operator names and folders as `matrix_element_analysis_fit_per_pz.ipynb`; this notebook does the forward frame only.

Run the cells from top to bottom. Section 4b asks before it saves a fit: answer `y` to save, anything else saves nothing.

Order: 0 settings, 1 summed 3pt and 2pt, 2 sGEVP $M_{\rm eff}(t)$, 3 fit, 4a check the fit, 4b save it, 5 single and double ratios.

## 0. Settings: data, operators of each pz, sGEVP t0

In [ ]:
import os
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"   #one BLAS thread: the many small fits of section 3 run faster;  only works before numpy is imported (a fresh kernel)
os.environ["OMP_NUM_THREADS"] = "1"
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
import gvar as gv
import lsqfit

#hadron
hadron = "proton"            #"pion" or "proton"
project = True               #proton only: True = positive-parity files, False = open Dirac files (none yet)
frame = "forward"            #this notebook does the forward frame only (q = 0)

#the data: every data set of this list is read, and they are joined along the cfg axis in this order (as in matrix_element_analysis_fit_per_pz.ipynb)
#  "dir": its folder;  "ncfg": the number of cfgs in the names of its 2pt files;  "ff_ncfg": the number of cfgs in the names of its FF files
#  "cfg_shift": added to its cfg numbers, so that the cfg numbers of two streams stay apart
#  "skip_cfg": cfgs of its 2pt files that are left out, e.g. a cfg that has no FF;  [] = none
data_sets = [{"name": "delta_stream_e", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta", "ncfg": 585, "ff_ncfg": 800, "cfg_shift": 0, "skip_cfg": []},
             {"name": "frontier_stream_d", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_frontier_streamd", "ncfg": 781, "ff_ncfg": 798, "cfg_shift": 100000, "skip_cfg": [3006]}]
ncfg = 0                     #the total number of cfgs: it goes in the names of the saved files
for data in data_sets:
    ncfg = ncfg + data["ncfg"] - len(data["skip_cfg"])
Lt = 96
Ls = 32
hyp = 8                      #hyp steps of the gluon operator

#sGEVP
sgevp_t0 = 2                 #GEVP reference time t0, the same at every pz
meff_below_t0 = True         #True = M_eff(t) also at t < t0, where the ground state is the smallest eigenvalue (HadStruc use t from 2 with t0 6-9);
                             #False = M_eff(t) only from t = t0 on.  At t0 = 2 both give the same M_eff from t = 2 on

#the operators of each pz:  smear B = [4.5,2.25,0.8], U = [1.5,1.5,0];  gamma G5, G5G4, G3G5
#pion_default and proton_default are the bases of matrix_element_analysis_fit_per_pz.ipynb (its t0 and t_ref are not used here: the sGEVP has its own t0)
pion_default = {0: ["BG5G4", "UG5", "UG5G4"], 1: ["UG5", "UG5G4"], 2: ["BG5", "UG5G4", "UG3G5"], 3: ["UG3G5"], 4: ["UG3G5"], 5: ["BG5G4", "UG5G4"], 6: ["BG3G5", "UG3G5"]}
proton_default = {0: ["UG5", "UG5G4"], 1: ["UG5", "UG5G4"], 2: ["UG5G4", "UG3G5"], 3: ["UG5G4", "UG3G5"], 4: ["UG5G4", "UG3G5"], 5: ["UG5G4"], 6: ["BG3G5"]}
all_six = {0: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"], 1: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"], 2: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"],
           3: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"], 4: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"], 5: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"],
           6: ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"]}
sgevp_ops = "six"            #"default" = pion_default or proton_default;  "six" = all six operators at every pz
#the choice the cells below use, and its name:  everything goes in the folder AI_assisted_matrix_element_analysis/{sgevp_tag}, in subfolders that start with sgevp_
#the default choice shares its folder with matrix_element_analysis_fit_per_pz.ipynb (pion_default, proton_default), so the ratio-fit results are next to these
if sgevp_ops == "default" and hadron == "pion":
    sgevp_choice = pion_default
    sgevp_tag = "pion_default"
elif sgevp_ops == "default":
    sgevp_choice = proton_default
    sgevp_tag = "proton_default"
else:
    sgevp_choice = all_six
    sgevp_tag = f"{hadron}_all_six"
op_basis = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]    #the six operators of the files, (smear, gamma):  smear 0 = B, 1 = U;  gamma 0 = G5, 1 = G5G4, 2 = G3G5
smear_names = ["B", "U"]
gamma_names = ["G5", "G5G4", "G3G5"]
op_labels = [smear_names[s] + gamma_names[g] for s, g in op_basis]     #["BG5", "BG5G4", ...]

#directories and names
analysis_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis"
save_dir = os.path.join(analysis_dir, sgevp_tag)
pt2_name = "2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_{frame}_q{qx}_{qy}_{qz}_{pt2_type}_fb_tsep20_3072src_ncfg{ncfg}_{hadron}.h5"
ff_name = "FF_hyp0-8_w0-9_q{qx}_{qy}_{qz}_ncfg{ncfg}.h5"
if meff_below_t0:
    meff_mode = "allt"       #in the names of the saved files
else:
    meff_mode = "fromt0"
#the name shared by every saved file of this choice and these settings
sgevp_name = f"forward_q0_0_0_{sgevp_tag}_t0{sgevp_t0}{meff_mode}_hyp{hyp}_ncfg{ncfg}"

assert frame == "forward", "this notebook does the forward frame only"
assert sgevp_ops in ["default", "six"], f"sgevp_ops = {sgevp_ops}: use default or six"
for pz in sgevp_choice:
    for name in sgevp_choice[pz]:
        assert name in op_labels, f"pz = {pz}: {name} is not one of {op_labels}"
if hadron == "pion":
    hadron_tag = "pion"
elif project:
    hadron_tag = "proton_projected"
else:
    hadron_tag = "proton"
c_sign = {"pion": -1, "proton": +1}[hadron]      #to make Cij positive definite, the pion is multiplied by -1


#======== functions used by several cells (the same as in matrix_element_analysis_fit_per_pz.ipynb) ========
def pt2_file(data, q_i, pt2_type):
    #the 2pt file of one data set of data_sets
    return os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type=pt2_type, ncfg=data["ncfg"], hadron=hadron_tag))


def read_joined(q_i, pt2_type, name):
    #one dataset of the 2pt files that has an entry per cfg, e.g. "pt2_forward", "t_src_list", "cfg_list":  read from every data set and joined along the cfg axis
    parts = []
    for data in data_sets:
        file = pt2_file(data, q_i, pt2_type)
        assert os.path.exists(file), f"no such file: {file}"
        with h5py.File(file, "r") as f:
            part = f[name][()]
            cfg_of_file = f["cfg_list"][()]
        assert part.shape[0] == data["ncfg"], f"{file}: {part.shape[0]} cfgs, but data_sets says {data['ncfg']}"
        assert np.isin(data["skip_cfg"], cfg_of_file).all(), f"{file}: a cfg of skip_cfg is not in this file"
        part = part[~np.isin(cfg_of_file, data["skip_cfg"])]     #leave out the cfgs of skip_cfg
        if name == "cfg_list":
            part = part + data["cfg_shift"]     #the cfg numbers of two streams stay apart
        parts.append(part)
    return np.concatenate(parts)


def read_shared(q_i, pt2_type, name):
    #one dataset of the 2pt files that has to be the same in every data set, e.g. "pz_list", "smear_list"
    for data in data_sets:
        file = pt2_file(data, q_i, pt2_type)
        assert os.path.exists(file), f"no such file: {file}"
        with h5py.File(file, "r") as f:
            value = f[name][()]
        if data is data_sets[0]:
            first = value
        assert np.array_equal(value, first), f"{name} of {data['name']} is not the same as in {data_sets[0]['name']}"
    return first


def jk(data, jk_axis=0):
    #jackknife mean and error
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    #per-cfg data (cfg, ...) -> jackknife samples (jk, ...)
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def op_matrix(pt2, basis):
    #(cfg, smear_snk, smear_src, gamma_snk, gamma_src, ...) -> (cfg, op_snk, op_src, ...)
    n_op = len(basis)
    Cij = np.zeros(pt2.shape[:1] + (n_op, n_op) + pt2.shape[5:], dtype=complex)
    for a, (s_snk, g_snk) in enumerate(basis):
        for b, (s_src, g_src) in enumerate(basis):
            Cij[:, a, b] = pt2[:, s_snk, s_src, g_snk, g_src]
    return Cij


def sgevp_fit_path(pz):
    #the saved fit of one pz: one file per pz, replaced when you save that pz again;  what the fit used is written inside the file
    return os.path.join(save_dir, "sgevp_fit_data", "forward", f"sgevp_fit_{sgevp_name}_{hadron_tag}_pz{pz}.h5")


#---- the folder of this choice:  another sgevp_choice with the same sgevp_tag stops here, so two choices cannot get mixed in one folder
choice_text = ""
for pz in sgevp_choice:
    choice_text = choice_text + f"pz {pz}: {' '.join(sgevp_choice[pz])}\n"
os.makedirs(save_dir, exist_ok=True)
choice_file = os.path.join(save_dir, "sgevp_choice.txt")
if not os.path.exists(choice_file):
    with open(choice_file, "w") as f:
        f.write(choice_text)
with open(choice_file, "r") as f:
    assert f.read() == choice_text, f"{choice_file} holds another sgevp_choice:  change sgevp_tag, or set sgevp_choice back"
for data in data_sets:
    print(f"data set {data['name']}: {data['ncfg']} cfg, {len(data['skip_cfg'])} left out, cfg numbers + {data['cfg_shift']}, folder {data['dir']}")
print("hadron", hadron_tag, "|", ncfg, "cfg in total | hyp", hyp, "| sGEVP t0", sgevp_t0, "| M_eff also below t0:", meff_below_t0)
print("operators", sgevp_tag, "| folder", save_dir)
print(choice_text)

## 1. Summed 3pt K(t) and 2pt C(t) of the six operators

In [ ]:
#======== summed 3pt K(t) and 2pt C(t) of the six operators, for every jackknife sample;  kept in memory, nothing is saved ========
#K_ab(t) = sum over tau = 1 .. t-1 of [<C_ab(t) O(tau)> - <C_ab(t)><O(tau)>]:  the 3pt of method 1 (as section 3 of matrix_element_analysis_fit_per_pz.ipynb) summed over the operator time
#  per source: the operator summed over the slices strictly between source and sink, times the 2pt of that source;  then the mean over the sources
#  vacuum term: the jackknife mean of the 2pt times the jackknife mean of the summed operator, over the same slices
#C_ab(t): the 2pt matrix, mean over the sources, its Hermitian part;  a = sink operator, b = source operator
half_sign = {"forward": +1, "backward": -1}     #operator at t_src + tau (forward half), t_src - tau (backward half)
q0 = (0, 0, 0)
time_start = time.time()
cfg_list = read_joined(q0, "C3input", "cfg_list")          #(cfg,): the cfgs of all data sets, cfg_shift added
pz_list = read_shared(q0, "C3input", "pz_list")             #(pz,)
smear_list = read_shared(q0, "C3input", "smear_list")       #(smear, 3): rho_T, rho_z, boost fraction
t_src_list = read_joined(q0, "C3input", "t_src_list")       #(cfg, t_src): the slice of each source, shifts per cfg
n_cfg, n_tsrc = t_src_list.shape
assert len(np.unique(cfg_list)) == len(cfg_list), "two data sets have a cfg number in common: give one of them another cfg_shift"

#---- the gluon operator O3 = TXTX + TYTY - 2 XYXY:  FF (cfg, munu, rhosig, hyp, w, t), munu order XY, YZ, XZ, XT, YT, ZT
#every data set has its own FF file;  the cfgs of its 2pt file are looked up in it by their cfg number, then the data sets are joined in the same order as the 2pt
O_parts = []
for data in data_sets:
    with h5py.File(pt2_file(data, q0, "C3input"), "r") as f:
        cfg_of_set = f["cfg_list"][()]              #the 2pt cfgs of this data set, its own numbers without cfg_shift
    cfg_of_set = cfg_of_set[~np.isin(cfg_of_set, data["skip_cfg"])]     #leave out the cfgs of skip_cfg
    FF_file = os.path.join(data["dir"], ff_name.format(qx=0, qy=0, qz=0, ncfg=data["ff_ncfg"]))
    print("reading", FF_file)
    assert os.path.exists(FF_file), f"no such file: {FF_file}"
    with h5py.File(FF_file, "r") as f:
        FF_cfg = f["cfg_list"][()]
        assert np.isin(cfg_of_set, FF_cfg).all(), f"{data['name']}: some 2pt cfgs are not in the FF file"
        idx = np.searchsorted(FF_cfg, cfg_of_set)   #the 2pt cfgs are not the first FF cfgs
        assert np.array_equal(FF_cfg[idx], cfg_of_set)
        ihyp = list(f["hyp_list"][()]).index(hyp)
        if data is data_sets[0]:
            w_list = f["w_list"][()]
        assert np.array_equal(f["w_list"][()], w_list), f"{data['name']}: other w than {data_sets[0]['name']}"
        TXTX = f["FF"][idx, 3, 3, ihyp]
        TYTY = f["FF"][idx, 4, 4, ihyp]
        XYXY = f["FF"][idx, 0, 0, ihyp]
    O_parts.append(TXTX + TYTY - 2 * XYXY)          #(cfg of this set, w, t)
O = np.concatenate(O_parts)                         #(cfg, w, t)

#---- C and K of each half
i_cfg = np.arange(n_cfg)[:, None, None]
C_jk = {}       #C_jk[half]: (jk, pz, t, op, op), c_sign in, Hermitian part
K_jk = {}       #K_jk[half]: (jk, w, pz, t, op, op), c_sign in
for half in half_sign:
    for data in data_sets:
        print("reading", pt2_file(data, q0, "C3input"), half)
    raw = read_joined(q0, "C3input", f"pt2_{half}")             #(cfg, smear, smear, gamma, gamma, t_src, pz, t)
    Cij = op_matrix(raw, op_basis)                              #(cfg, op, op, t_src, pz, t)
    del raw
    n_tsep = Cij.shape[-1]
    C_cfg = Cij.mean(axis=3)                                    #(cfg, op, op, pz, t): mean over the sources
    C = np.moveaxis(c_sign * jackknife(C_cfg), (1, 2), (-2, -1))                #(jk, pz, t, op, op)
    C_jk[half] = 0.5 * (C + np.conj(np.swapaxes(C, -1, -2)))                    #Hermitian part
    t_op = (t_src_list[:, :, None] + half_sign[half] * np.arange(n_tsep)) % Lt  #(cfg, t_src, tau): the slice of the operator
    O_tau = O[i_cfg, :, t_op]                                   #(cfg, t_src, tau, w): numpy puts w last here
    O_sum = np.zeros_like(O_tau)                                #(cfg, t_src, t, w): O summed over tau = 1 .. t-1
    for t in range(n_tsep):
        O_sum[:, :, t] = O_tau[:, :, 1:t].sum(axis=2)
    connected = np.einsum("cabspt,cstw->cabwpt", Cij, O_sum, optimize=True) / n_tsrc            #(cfg, op, op, w, pz, t): source by source, mean over the sources
    vacuum = np.einsum("cabpt,ctw->cabwpt", jackknife(C_cfg), jackknife(O_sum.mean(axis=1)), optimize=True)    #(jk, op, op, w, pz, t)
    K_jk[half] = np.moveaxis(c_sign * (jackknife(connected) - vacuum), (1, 2), (-2, -1))       #(jk, w, pz, t, op, op)
    del Cij, C_cfg, C, O_tau, O_sum, connected, vacuum

#check: the summed ratio K/C of one operator at t 8, w 0, both halves (single-operator numbers, before any GEVP)
for half in half_sign:
    a = op_labels.index(sgevp_choice[0][0])
    m, e = jk(K_jk[half][:, 0, :, 8, a, a].real / C_jk[half][:, :, 8, a, a].real)
    print(f"{half}, {op_labels[a]}: K(8)/C(8) at w 0, pz {' '.join(str(int(p)) for p in pz_list)}:", " ".join(f"{x:+.3f}({y * 1e3:.0f})" for x, y in zip(m, e)))
print(f"{n_cfg} cfg, {len(pz_list)} pz, {len(w_list)} w, t 0-{n_tsep - 1};  {time.time() - time_start:.0f} s")

## 2. sGEVP: M_eff(t) of each pz, saved with a plot

In [ ]:
#======== sGEVP:  R(t) = u^dag K(t) u / lambda(t, t0) - u^dag K(t0) u,   M_eff(t) = R(t+1) - R(t);  saves M_eff and its plot ========
#u, lambda: the ground state of the GEVP C(t) u = lambda C(t0) u, solved again at every t and for every jackknife sample, with u^dag C(t0) u = 1
#  t > t0: the ground state has the largest eigenvalue, lambda = exp(-E0 (t - t0)) < 1
#  t < t0 (meff_below_t0 = True): the ground state has the smallest eigenvalue, lambda = exp(E0 (t0 - t)) > 1;  R(t) is the same formula
#  R(t0) = 0.  Each half has its own GEVP;  R is the mean of the two halves.  Re[u^dag K u] = u^dag (K + K^dag)/2 u: the Hermitian part of K
meff_plot_w = [0, 2, 4, 6, 8]       #the w drawn in the plot
meff_plot_err_max = 0.3             #plot: points with a larger error are left out

n_jk = C_jk["forward"].shape[0]
n_pz = len(pz_list)
n_w = len(w_list)
if meff_below_t0:
    R_t_list = list(range(1, sgevp_t0)) + list(range(sgevp_t0 + 1, n_tsep))    #every t but t0
else:
    R_t_list = list(range(sgevp_t0 + 1, n_tsep))
meff_jk = np.full((n_jk, n_pz, n_w, n_tsep), np.nan)       #(jk, pz, w, t): M_eff(t) = R(t+1) - R(t);  NaN where R is not made
for ipz, pz in enumerate(pz_list):
    use = [op_labels.index(name) for name in sgevp_choice[int(pz)]]           #the operators of this pz
    R = np.zeros((n_jk, n_w, n_tsep))                                           #(jk, w, t): mean of the two halves
    for half in half_sign:
        Cu = C_jk[half][:, ipz][:, :, use][:, :, :, use]                        #(jk, t, n, n)
        Ku = K_jk[half][:, :, ipz][:, :, :, use][:, :, :, :, use]               #(jk, w, t, n, n)
        L_inv = np.linalg.inv(np.linalg.cholesky(Cu[:, sgevp_t0]))              #(jk, n, n):  C(t0) = L L^dag
        L_inv_dag = np.conj(np.swapaxes(L_inv, -1, -2))
        R_half = np.full((n_jk, n_w, n_tsep), np.nan)                           #(jk, w, t)
        R_half[:, :, sgevp_t0] = 0
        for t in R_t_list:
            M = L_inv @ Cu[:, t] @ L_inv_dag                                    #(jk, n, n): same eigenvalues as the GEVP
            M = 0.5 * (M + np.conj(np.swapaxes(M, -1, -2)))
            eig_val, eig_vec = np.linalg.eigh(M)                                #eigenvalues in increasing order
            if t > sgevp_t0:
                k = -1                                                          #largest eigenvalue
            else:
                k = 0                                                           #smallest eigenvalue
            u = (L_inv_dag @ eig_vec[:, :, k][:, :, None])[:, :, 0]             #(jk, n): the GEVP vector, u^dag C(t0) u = 1
            uKu = np.einsum("ja,jwab,jb->jw", u.conj(), Ku[:, :, t], u).real    #(jk, w)
            uK0u = np.einsum("ja,jwab,jb->jw", u.conj(), Ku[:, :, sgevp_t0], u).real
            R_half[:, :, t] = uKu / eig_val[:, k][:, None] - uK0u
        R = R + 0.5 * R_half
    meff_jk[:, ipz, :, :-1] = R[:, :, 1:] - R[:, :, :-1]

#---- print: M_eff at w 0
meff_mean, meff_err = jk(meff_jk)                           #(pz, w, t)
print("M_eff(t) at w 0, operators per pz:", ", ".join(f"pz {pz}: {' '.join(sgevp_choice[int(pz)])}" for pz in pz_list))
print(f"{'t':>3s} " + " ".join(f"{'pz = ' + str(pz):>13s}" for pz in pz_list))
for t in range(1, 15):
    print(f"{t:3d} " + " ".join(f"{str(gv.gvar(meff_mean[i, 0, t], meff_err[i, 0, t])) if np.isfinite(meff_mean[i, 0, t]) else '-':>13s}" for i in range(n_pz)))

#---- save M_eff
meff_file = os.path.join(save_dir, "sgevp_meff_data", "forward", f"sgevp_meff_{sgevp_name}_{hadron_tag}.h5")
if os.path.exists(meff_file):
    print("replacing", meff_file)
os.makedirs(os.path.dirname(meff_file), exist_ok=True)
with h5py.File(meff_file, "w") as f:
    f["meff_jk"] = meff_jk                                  #(jk, pz, w, t)
    f["t_list"] = np.arange(n_tsep)
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["cfg_list"] = cfg_list
    f["sgevp_t0"] = sgevp_t0
    f["meff_below_t0"] = meff_below_t0
    f["sgevp_tag"] = sgevp_tag
    f["operators"] = [" ".join(sgevp_choice[int(pz)]) for pz in pz_list]       #(pz,): the operators of each pz in words
    f["op_labels"] = op_labels
    f["smear_list"] = smear_list
    f["hyp"] = hyp
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["data_sets"] = [data["name"] for data in data_sets]
    f["data_set_ncfg"] = [data["ncfg"] for data in data_sets]
    f["data_set_cfg_shift"] = [data["cfg_shift"] for data in data_sets]
    f["data_set_skip_cfg"] = [" ".join(str(c) for c in data["skip_cfg"]) for data in data_sets]
print("saved", meff_file)

#---- plot: M_eff(t) of every pz, a few w
meff_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#52514e", "#a8a79f", "#000000"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for ipz, pz in enumerate(pz_list):
    ax = axes.ravel()[ipz]
    for iw, w in enumerate(w_list):
        if w in meff_plot_w:
            ok = np.isfinite(meff_mean[ipz, iw]) & (meff_err[ipz, iw] < meff_plot_err_max)
            t_ok = np.arange(n_tsep)[ok]
            ax.errorbar(t_ok + 0.06 * (iw - 4), meff_mean[ipz, iw, ok], meff_err[ipz, iw, ok], color=meff_colors[iw], marker="o", ms=3.5, lw=0, elinewidth=1, capsize=1.5, label=f"w = {w}")
    ax.axvline(sgevp_t0, color="#a8a79f", lw=0.8, ls=":")
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"pz = {pz}: {' '.join(sgevp_choice[int(pz)])}", fontsize=10, loc="left")
    ax.set_xlabel("t / a")
    ax.set_ylabel(r"$M_{\rm eff}(t)$")
axes.ravel()[0].legend(frameon=False, fontsize=8)
axes.ravel()[-1].axis("off")
fig.suptitle(f"{hadron_tag}, forward, {ncfg} cfg, HYP {hyp}: sGEVP M_eff(t), t0 = {sgevp_t0} (dotted line){', also below t0' if meff_below_t0 else ''};  points with error > {meff_plot_err_max} left out", fontsize=11)
fig.tight_layout()
meff_plot_file = os.path.join(save_dir, "sgevp_meff_plots", "forward", f"sgevp_meff_{sgevp_name}_{hadron_tag}.png")
os.makedirs(os.path.dirname(meff_plot_file), exist_ok=True)
fig.savefig(meff_plot_file, dpi=150)
plt.show()
print("saved", meff_plot_file)

## 3. Fit $M_{\rm eff}(t) = A + B\, t\, e^{-\Delta E\, t}$: every fit parameter is here

In [ ]:
#======== fit M_eff(t) = A + B t exp(-dE t):  every fit parameter is here;  this cell saves nothing: section 4a shows the fits, section 4b saves them ========
#A = the matrix element M(pz, w);  B t exp(-dE t) = the excited states that are left;  dE: one for all w of one fit, or one per w (fit_shared_gap)
fit_pz_run = [0, 1, 2, 3, 4, 5, 6]     #the pz to fit
#fit window in t of each pz: first t, last t (t = source-sink separation;  M_eff(t) = R(t+1) - R(t))
#these give the six-operator double ratio with one gap for all w (proton, 1365 cfg, sGEVP t0 2, HYP 8)
fit_t = {0: (2, 12), 1: (2, 12), 2: (2, 9), 3: (2, 8), 4: (2, 7), 5: (2, 6), 6: (2, 6)}
#used before with the default operators: {0: (2, 12), 1: (2, 12), 2: (2, 10), 3: (2, 10), 4: (2, 9), 5: (2, 8), 6: (2, 8)}
fit_procedure = "one_fit"       #"one_fit" = all ten w in one fit with one dE;
                                #"paper" = w 0 fitted alone first, then w 1-9 together with one dE whose prior is the w 0 value with fit_dE_width_factor times its jackknife error (Khan et al.)
fit_dE_width_factor = 3         #"paper" only
fit_shared_gap = True           #True = one dE for all w of one fit;  False = every w has its own dE (still one fit with the full covariance across w;
                                #       "paper": each dE of w 1-9 then has the prior from the w 0 fit)
fit_A_sign = "free"             #"free" = prior fit_A_prior on A;  "negative" = A = -exp(a), prior log(-A) = 0(10), so A cannot change sign
fit_A_prior = (0.0, 10.0)       #(center, width), "free" only
fit_B_prior = (0.0, 10.0)       #(center, width)
fit_dE_prior = (0.5, 0.5)       #(center, width), log-normal: the fit varies log(dE)
fit_dE_starts = [0.3, 0.5, 0.8, 1.2]    #the fit of the jackknife mean starts from each dE (A = last point, B from the first point) and keeps the lowest chi2;  every jackknife fit starts from it
fit_svdcut = 1e-12              #lsqfit's default
fit_maxit = 20000
fit_tol = 1e-8
fit_fitter = "scipy_least_squares"
#fixed: the covariance of the points is the jackknife covariance of the mean, made once from all samples and used for the fit of the mean and for every jackknife fit;
#       all t of the window and all w of one fit are correlated (full covariance)

assert fit_procedure in ["one_fit", "paper"], f"fit_procedure = {fit_procedure}: use one_fit or paper"
assert fit_A_sign in ["free", "negative"], f"fit_A_sign = {fit_A_sign}: use free or negative"
if not gv.BufferDict.has_distribution("neglog"):       #a name for gvar, like its own "log":  the fit varies neglog(A), and p["A"] = -exp(neglog(A))
    gv.BufferDict.add_distribution("neglog", lambda x: -gv.exp(x))


def meff_model(x, p):
    #x["t"], x["iw"]: time and w index (inside this fit) of every point
    if fit_shared_gap:
        dE = p["dE"][0]
    else:
        dE = p["dE"][x["iw"]]
    return p["A"][x["iw"]] + p["B"][x["iw"]] * x["t"] * gv.exp(-dE * x["t"])


def meff_prior(n_group, dE_prior):
    #n_group = how many w are in this fit
    prior = gv.BufferDict()
    if fit_A_sign == "negative":
        prior["neglog(A)"] = gv.log(-gv.gvar(np.full(n_group, -1.0), np.full(n_group, 10.0)))    #read it as fit.p["A"]
    else:
        prior["A"] = gv.gvar(np.full(n_group, fit_A_prior[0]), np.full(n_group, fit_A_prior[1]))
    prior["B"] = gv.gvar(np.full(n_group, fit_B_prior[0]), np.full(n_group, fit_B_prior[1]))
    if fit_shared_gap:
        n_gap = 1
    else:
        n_gap = n_group
    prior["log(dE)"] = gv.log(gv.gvar(np.full(n_gap, dE_prior[0]), np.full(n_gap, dE_prior[1])))     #read it as fit.p["dE"]
    return prior


if fit_procedure == "one_fit":
    w_groups = [list(range(n_w))]                   #every w in one fit
else:
    w_groups = [[0], list(range(1, n_w))]           #w 0 alone, then the others
fit_param_names = ["A", "B", "dE"]
param_jk = {}           #param_jk[pz][name]: (jk, w);  dE is the same at every w of one fit
param_central = {}      #param_central[pz][name]: (w,), the fit of the jackknife mean
A_lsqfit_err = {}       #A_lsqfit_err[pz]: (w,), the lsqfit error of A in the fit of the mean
fit_chi2 = {}           #fit_chi2[pz]: (w,), the chi2 of the fit that holds this w
fit_dof = {}            #fit_dof[pz]: (w,)
fit_dE_prior_used = {}  #fit_dE_prior_used[pz]: (w, 2), the dE prior of the fit that holds this w
fit_start_chi2 = {}     #fit_start_chi2[pz]: (group, start), chi2 of the fit of the mean from each start
fit_n_not_converged = {}    #how many fits stopped at fit_maxit before converging;  lsqfit gives no warning for this
fit_n_total = {}
fit_points_t = {}       #fit_points_t[pz]: the t of the window
fit_y_jk = {}           #fit_y_jk[pz]: (jk, w, n_t), the fitted M_eff
check_fig = {}          #the figures of section 4a;  emptied here, so an old figure cannot be saved with a new fit
for pz in fit_pz_run:
    time_start = time.time()
    ipz = list(pz_list).index(pz)
    first, last = fit_t[pz]
    t_points = np.arange(first, last + 1)
    y_all = meff_jk[:, ipz][:, :, t_points]         #(jk, w, n_t)
    assert not np.isnan(y_all).any(), f"pz = {pz}: M_eff has no value at some t of the window {first}-{last};  meff_below_t0 = False needs first t >= t0"
    fit_points_t[pz] = t_points
    fit_y_jk[pz] = y_all
    param_jk[pz] = {}
    param_central[pz] = {}
    for name in fit_param_names:
        param_jk[pz][name] = np.zeros((n_jk, n_w))
        param_central[pz][name] = np.zeros(n_w)
    A_lsqfit_err[pz] = np.zeros(n_w)
    fit_chi2[pz] = np.zeros(n_w)
    fit_dof[pz] = np.zeros(n_w, dtype=int)
    fit_dE_prior_used[pz] = np.zeros((n_w, 2))
    fit_start_chi2[pz] = np.zeros((len(w_groups), len(fit_dE_starts)))
    fit_n_not_converged[pz] = 0
    fit_n_total[pz] = 0

    for ig, group in enumerate(w_groups):
        #the dE prior of this fit
        if fit_procedure == "paper" and ig == 1:
            dE_mean_w0, dE_err_w0 = jk(param_jk[pz]["dE"][:, 0])
            dE_prior = (param_central[pz]["dE"][0], fit_dE_width_factor * dE_err_w0)
        else:
            dE_prior = fit_dE_prior
        #the data of this fit:  y (jk, n_group * n_t), w after w
        n_group = len(group)
        y = y_all[:, group].reshape(n_jk, n_group * len(t_points))
        x = {"t": np.tile(t_points, n_group).astype(float), "iw": np.repeat(np.arange(n_group), len(t_points))}
        y_mean = y.mean(axis=0)
        d = y - y_mean
        cov = (n_jk - 1) / n_jk * (d.T @ d)          #jackknife covariance of the mean
        y_mean_w = y_mean.reshape(n_group, len(t_points))

        gv.switch_gvar()            #keeps memory flat when the cell is run again
        #the fit of the jackknife mean, from each start
        best = None
        for i_start, dE_start in enumerate(fit_dE_starts):
            if fit_shared_gap:
                p0 = {"log(dE)": np.log([dE_start])}
            else:
                p0 = {"log(dE)": np.log(np.full(n_group, dE_start))}
            p0["B"] = (y_mean_w[:, 0] - y_mean_w[:, -1]) / (t_points[0] * np.exp(-dE_start * t_points[0]))
            if fit_A_sign == "negative":
                p0["neglog(A)"] = np.log(np.maximum(-y_mean_w[:, -1], 1e-3))
            else:
                p0["A"] = y_mean_w[:, -1]
            fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_mean, cov)), fcn=meff_model, prior=meff_prior(n_group, dE_prior), p0=p0,
                                       svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
            fit_start_chi2[pz][ig, i_start] = fit.chi2
            fit_n_total[pz] += 1
            if best is None or fit.chi2 < best.chi2:
                best = fit
        fit_n_not_converged[pz] += int(best.stopping_criterion == 0)
        p0_jk = {}
        for name in best.pmean:
            if name in ["A", "neglog(A)", "B", "log(dE)"]:
                p0_jk[name] = np.array(best.pmean[name])
        for k, iw in enumerate(group):
            param_central[pz]["A"][iw] = best.pmean["A"][k]
            param_central[pz]["B"][iw] = best.pmean["B"][k]
            if fit_shared_gap:
                param_central[pz]["dE"][iw] = best.pmean["dE"][0]
            else:
                param_central[pz]["dE"][iw] = best.pmean["dE"][k]
            A_lsqfit_err[pz][iw] = best.psdev["A"][k]
            fit_chi2[pz][iw] = best.chi2
            fit_dof[pz][iw] = best.dof
            fit_dE_prior_used[pz][iw] = dE_prior
        #every jackknife sample: same covariance, same prior, started from the fit of the mean
        for j in range(n_jk):
            fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y[j], cov)), fcn=meff_model, prior=meff_prior(n_group, dE_prior), p0=p0_jk,
                                       svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
            for k, iw in enumerate(group):
                param_jk[pz]["A"][j, iw] = fit.pmean["A"][k]
                param_jk[pz]["B"][j, iw] = fit.pmean["B"][k]
                if fit_shared_gap:
                    param_jk[pz]["dE"][j, iw] = fit.pmean["dE"][0]
                else:
                    param_jk[pz]["dE"][j, iw] = fit.pmean["dE"][k]
            fit_n_not_converged[pz] += int(fit.stopping_criterion == 0)
            fit_n_total[pz] += 1
        gv.restore_gvar()

    A_mean, A_err = jk(param_jk[pz]["A"])
    dE_mean, dE_err = jk(param_jk[pz]["dE"])
    groups_text = ""
    for group in w_groups:
        if len(group) == 1:
            w_text = f"w {group[0]}"
        else:
            w_text = f"w {group[0]}-{group[-1]}"
        if fit_shared_gap or len(group) == 1:
            dE_text = str(gv.gvar(param_central[pz]["dE"][group[0]], dE_err[group[0]]))
        else:
            dE_text = "one per w, below"
        groups_text = groups_text + f"{w_text}: chi2/dof {fit_chi2[pz][group[0]] / fit_dof[pz][group[0]]:.2f}, dE {dE_text};  "
    print(f"pz {pz} ({' '.join(sgevp_choice[pz])}), t {first}-{last}, {fit_procedure}: {groups_text}A(w 0) {gv.gvar(param_central[pz]['A'][0], A_err[0])};"
          f"  {fit_n_not_converged[pz]} of {fit_n_total[pz]} fits stopped at maxit;  {time.time() - time_start:.0f} s")
    if not fit_shared_gap:
        print(f"      dE per w: " + " ".join(str(gv.gvar(param_central[pz]["dE"][iw], dE_err[iw])) for iw in range(n_w)))

## 4a. Check the fits: M_eff and the bands

In [ ]:
#======== check the fits of section 3: one figure per pz, one panel per w;  nothing is saved here ========
#filled points: in the fit window;  open grey points: outside;  band: A + B t exp(-dE t) with its jackknife error over the window
#mean pull = (point - curve of the fit of the mean) / point error, averaged over the window:  the band has to pass through the points (|mean pull| well below 1)
check_t_max = 14                    #points drawn up to this t
check_err_max = 0.3                 #points with a larger error are left out of the plot

for pz in fit_pz_run:
    ipz = list(pz_list).index(pz)
    first, last = fit_t[pz]
    fig, axes = plt.subplots(2, 5, figsize=(17, 6.6))
    pulls = np.zeros(n_w)
    for iw, w in enumerate(w_list):
        ax = axes.ravel()[iw]
        t_all = np.arange(1, check_t_max + 1)
        m, e = jk(meff_jk[:, ipz, iw, t_all])
        inside = (t_all >= first) & (t_all <= last)
        ok = np.isfinite(m) & (e < check_err_max)
        ax.errorbar(t_all[ok & inside], m[ok & inside], e[ok & inside], color="#2a78d6", marker="o", ms=4, lw=0, elinewidth=1, capsize=1.5)
        ax.errorbar(t_all[ok & ~inside], m[ok & ~inside], e[ok & ~inside], color="#a8a79f", marker="o", mfc="white", ms=4, lw=0, elinewidth=1, capsize=1.5)
        #band over the window
        t_band = np.linspace(first, last, 80)
        central = param_central[pz]["A"][iw] + param_central[pz]["B"][iw] * t_band * np.exp(-param_central[pz]["dE"][iw] * t_band)
        samples = param_jk[pz]["A"][:, iw, None] + param_jk[pz]["B"][:, iw, None] * t_band[None, :] * np.exp(-param_jk[pz]["dE"][:, iw, None] * t_band[None, :])
        band_mean, band_err = jk(samples)
        ax.fill_between(t_band, central - band_err, central + band_err, color="#eb6834", alpha=0.25, lw=0)
        ax.plot(t_band, central, color="#eb6834", lw=1)
        #mean pull over the window
        t_win = fit_points_t[pz]
        curve = param_central[pz]["A"][iw] + param_central[pz]["B"][iw] * t_win * np.exp(-param_central[pz]["dE"][iw] * t_win)
        y_m, y_e = jk(fit_y_jk[pz][:, iw])
        pulls[iw] = np.mean((y_m - curve) / y_e)
        A_m, A_e = jk(param_jk[pz]["A"][:, iw])
        ax.set_title(f"w = {w}:  A = {gv.gvar(param_central[pz]['A'][iw], float(A_e))},  mean pull {pulls[iw]:+.2f}", fontsize=9, loc="left")
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel("t / a", fontsize=9)
    chi2_text = "; ".join(f"w {group[0] if len(group) == 1 else f'{group[0]}-{group[-1]}'} chi2/dof {fit_chi2[pz][group[0]] / fit_dof[pz][group[0]]:.2f}" for group in w_groups)
    fig.suptitle(f"{hadron_tag}, forward, {ncfg} cfg, HYP {hyp}, pz {pz} ({' '.join(sgevp_choice[pz])}), sGEVP t0 {sgevp_t0}: M_eff(t) and A + B t exp(-dE t), t {first}-{last}, {fit_procedure}, A {fit_A_sign}, {'one dE' if fit_shared_gap else 'dE per w'};  "
                 f"{chi2_text};  largest |mean pull| {np.abs(pulls).max():.2f}", fontsize=10)
    fig.tight_layout()
    check_fig[pz] = fig
    plt.show()
    print(f"pz {pz}: mean pull per w:", " ".join(f"{p:+.2f}" for p in pulls))

## 4b. Save the fits?

In [ ]:
#======== save the fits of section 3?  it asks for each pz, and saves that pz only if you answer y ========
#one file per pz, replaced when you save that pz again;  the file states the fitted data and every setting;  section 5 reads these files
for pz in fit_pz_run:
    assert pz in check_fig, f"pz = {pz} has no figure: run section 4a after section 3, then this cell"
for pz in fit_pz_run:
    answer = input(f"save the sGEVP fit of pz = {pz}? [y/N] ")
    if answer.strip().lower() in ("y", "yes"):
        fit_file = sgevp_fit_path(pz)
        if os.path.exists(fit_file):
            print("replacing", fit_file)
        os.makedirs(os.path.dirname(fit_file), exist_ok=True)
        with h5py.File(fit_file, "w") as f:
            #results
            for name in fit_param_names:
                f[f"{name}_jk"] = param_jk[pz][name]            #(jk, w);  with fit_shared_gap, dE is the same at every w of one fit
                f[f"{name}_central"] = param_central[pz][name]  #(w,)
            f["A_lsqfit_err"] = A_lsqfit_err[pz]                #(w,)
            f["chi2"] = fit_chi2[pz]                            #(w,): chi2 of the fit that holds this w
            f["dof"] = fit_dof[pz]                              #(w,)
            f["start_chi2"] = fit_start_chi2[pz]                #(group, start)
            f["n_not_converged"] = fit_n_not_converged[pz]
            f["n_fits"] = fit_n_total[pz]
            f["fit_data_jk"] = fit_y_jk[pz]                     #(jk, w, n_t): the fitted M_eff
            f["fit_t"] = fit_points_t[pz]                       #(n_t,)
            #settings
            f["pz"] = pz
            f["operators"] = " ".join(sgevp_choice[pz])
            f["model"] = "sGEVP M_eff(t) = A + B t exp(-dE t)"
            f["meff_file"] = meff_file
            f["sgevp_t0"] = sgevp_t0
            f["meff_below_t0"] = meff_below_t0
            f["fit_window"] = fit_t[pz]                         #first t, last t
            f["fit_procedure"] = fit_procedure
            f["fit_dE_width_factor"] = fit_dE_width_factor
            f["fit_shared_gap"] = fit_shared_gap
            f["fit_A_sign"] = fit_A_sign
            f["fit_A_prior"] = fit_A_prior
            f["fit_B_prior"] = fit_B_prior
            f["fit_dE_prior"] = fit_dE_prior
            f["dE_prior_used"] = fit_dE_prior_used[pz]          #(w, 2): the dE prior of the fit that holds this w
            f["fit_dE_starts"] = fit_dE_starts
            f["svdcut"] = fit_svdcut
            f["maxit"] = fit_maxit
            f["tol"] = fit_tol
            f["fitter"] = fit_fitter
            #what the data are, as stated in the M_eff file
            with h5py.File(meff_file, "r") as f_meff:
                for name in ["pz_list", "w_list", "cfg_list", "sgevp_tag", "op_labels", "smear_list", "hyp", "hadron_tag", "ncfg",
                             "data_sets", "data_set_ncfg", "data_set_cfg_shift", "data_set_skip_cfg"]:
                    f_meff.copy(name, f)
        print("saved", fit_file)
        fit_plot_file = os.path.join(save_dir, "sgevp_fit_plots", "forward", os.path.basename(fit_file).replace(".h5", ".png"))
        os.makedirs(os.path.dirname(fit_plot_file), exist_ok=True)
        check_fig[pz].savefig(fit_plot_file, dpi=150, bbox_inches="tight")
        print("saved", fit_plot_file)
    else:
        print(f"pz = {pz}: not saved")

## 5. Single and double ratios: plots and h5 file

In [ ]:
#======== single and double ratios from the saved sGEVP fits: the plots and one h5 file;  only section 0 has to be run before this cell ========
#double ratio D(pz, w) = [M(pz, w) / M(pz, w_ref)] / [M(0, w) / M(0, w_ref)]
#single ratio S(pz, w) = M(pz, w) / M(0, w) * E(0) / E(pz),  E = the central E0 of a saved 2pt fit (of matrix_element_analysis_fit_per_pz.ipynb)
#M: A of the saved sGEVP fit of each pz (section 4b)
itd_pz_list = [0, 1, 2, 3, 4, 5, 6]     #the pz to include;  each needs a saved fit;  pz 0 is the reference
itd_w_ref = 0                       #double ratio: every matrix element is divided by its value at this w
itd_plot_err_max = 0.5              #leave out points with a larger error, so noisy points do not set the y range
#the saved 2pt fit for E (the single ratio):  this path is where matrix_element_analysis_fit_per_pz.ipynb saves it for the default bases;  None = no single ratio
itd_pt2_fit_file = os.path.join(analysis_dir, f"{hadron}_default", "2pt_fit_data", "forward", f"2pt_fit_forward_q0_0_0_{hadron}_default_fb_ncfg{ncfg}_{hadron_tag}.h5")
#the ratio-fit double ratio to compare with (drawn in grey, per pz):  where matrix_element_analysis_fit_per_pz.ipynb saves it;  None = no comparison
itd_compare_file = os.path.join(analysis_dir, f"{hadron}_default", "itd_data", "forward", f"itd_forward_q0_0_0_{hadron}_default_phase2pt_fb_hyp{hyp}_ncfg{ncfg}_refp000_wref0_{hadron_tag}.h5")

#---- read the saved fit of every pz
assert 0 in itd_pz_list, "pz 0 has to be in itd_pz_list, it is the reference"
itd_fit_file = {}
for pz in itd_pz_list:
    itd_fit_file[pz] = sgevp_fit_path(pz)
    assert os.path.exists(itd_fit_file[pz]), f"no saved sGEVP fit of pz = {pz}: fit it in section 3 and save it in section 4b, or take it out of itd_pz_list.  Looked for {itd_fit_file[pz]}"
with h5py.File(itd_fit_file[itd_pz_list[0]], "r") as f:
    w_values = f["w_list"][()]
    itd_cfg_list = f["cfg_list"][()]
n_jk_itd = len(itd_cfg_list)
n_w_itd = len(w_values)
n_pz_itd = len(itd_pz_list)
M_jk = np.zeros((n_jk_itd, n_w_itd, n_pz_itd))      #(jk, w, pz): A of every jackknife sample
for i, pz in enumerate(itd_pz_list):
    with h5py.File(itd_fit_file[pz], "r") as f:
        assert np.array_equal(f["cfg_list"][()], itd_cfg_list), f"pz = {pz}: the saved fit has other cfgs than pz = {itd_pz_list[0]}"
        assert np.array_equal(f["w_list"][()], w_values), f"pz = {pz}: the saved fit has other w than pz = {itd_pz_list[0]}"
        M_jk[:, :, i] = f["A_jk"][()]
i_ref = itd_pz_list.index(0)
M_ref_jk = M_jk[:, :, i_ref:i_ref + 1]              #(jk, w, 1)
iw_ref = list(w_values).index(itd_w_ref)
nu = 2 * np.pi / Ls * np.outer(w_values, itd_pz_list)                                               #(w, pz)
double_jk = (M_jk / M_jk[:, iw_ref:iw_ref + 1]) / (M_ref_jk / M_ref_jk[:, iw_ref:iw_ref + 1])       #(jk, w, pz)
double_mean, double_err = jk(double_jk)                                                             #(w, pz)

#---- the single ratio, if there is a saved 2pt fit of the same cfgs
make_single = itd_pt2_fit_file is not None and os.path.exists(itd_pt2_fit_file)
if make_single:
    E_central = np.zeros(n_pz_itd)                  #(pz,)
    with h5py.File(itd_pt2_fit_file, "r") as f:
        assert np.array_equal(f["cfg_list"][()], itd_cfg_list), "the 2pt fit has other cfgs than the sGEVP fits"
        pt2_pz_list = list(f["pz_list"][()])
        for i, pz in enumerate(itd_pz_list):
            E_central[i] = f["E_central"][pt2_pz_list.index(pz), 0]
    E_ref = E_central[i_ref]
    single_jk = M_jk / M_ref_jk * (E_ref / E_central)                                               #(jk, w, pz)
    single_mean, single_err = jk(single_jk)
else:
    print(f"no saved 2pt fit at {itd_pt2_fit_file}:  the single ratio is not made, only the double ratio")

#---- print: what went in, then the tables
print(f"{hadron_tag}, forward, {n_jk_itd} cfg, sGEVP t0 {sgevp_t0}{' (M_eff also below t0)' if meff_below_t0 else ''}, reference p = (0,0,0), w_ref {itd_w_ref}")
for pz in itd_pz_list:
    with h5py.File(itd_fit_file[pz], "r") as f:
        print(f"pz {pz}: {f['operators'].asstr()[()]}, t {f['fit_window'][0]}-{f['fit_window'][1]}, {f['fit_procedure'].asstr()[()]}, A {f['fit_A_sign'].asstr()[()]}, "
              f"{'one dE' if f['fit_shared_gap'][()] else 'dE per w'}")
tables = [("double ratio", double_mean, double_err)]
if make_single:
    tables = [("single ratio", single_mean, single_err)] + tables
for title, mean, err in tables:
    print(title)
    print(f"{'w':>3s} " + " ".join(f"{'pz = ' + str(pz):>12s}" for pz in itd_pz_list))
    for iw, w in enumerate(w_values):
        print(f"{w:3d} " + " ".join(f"{str(gv.gvar(mean[iw, i], err[iw, i])):>12s}" for i in range(n_pz_itd)))

#---- save
itd_name = f"itd_sgevp_{sgevp_name}_refp000_wref{itd_w_ref}_{hadron_tag}"
itd_file = os.path.join(save_dir, "sgevp_itd_data", "forward", itd_name + ".h5")
if os.path.exists(itd_file):
    print("replacing", itd_file)
os.makedirs(os.path.dirname(itd_file), exist_ok=True)
with h5py.File(itd_file, "w") as f:
    f["double_ratio_jk"] = double_jk        #(jk, w, pz)
    f["double_ratio_mean"] = double_mean    #(w, pz)
    f["double_ratio_err"] = double_err      #(w, pz)
    if make_single:
        f["single_ratio_jk"] = single_jk    #(jk, w, pz)
        f["single_ratio_mean"] = single_mean
        f["single_ratio_err"] = single_err
        f["E"] = E_central                  #(pz,): the central E0 of the 2pt fit
        f["E_ref"] = E_ref
        f["pt2_fit_file"] = itd_pt2_fit_file
    f["M_jk"] = M_jk                        #(jk, w, pz)
    f["nu"] = nu                            #(w, pz): 2 pi / Ls * pz * w
    f["w_ref"] = itd_w_ref
    f["pz_list"] = itd_pz_list
    f["w_list"] = w_values
    f["cfg_list"] = itd_cfg_list
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["hyp"] = hyp
    f["Ls"] = Ls
    for pz in itd_pz_list:
        #what the fit of this pz used:  f["choice/pz3/fit_window"] and so on
        f[f"choice/pz{pz}/sgevp_fit_file"] = itd_fit_file[pz]
        with h5py.File(itd_fit_file[pz], "r") as fit_in:
            for name in ["operators", "sgevp_t0", "meff_below_t0", "fit_window", "fit_procedure", "fit_shared_gap", "fit_A_sign", "fit_A_prior", "fit_B_prior", "fit_dE_prior", "svdcut"]:
                fit_in.copy(name, f, name=f"choice/pz{pz}/{name}")
    with h5py.File(itd_fit_file[itd_pz_list[0]], "r") as fit_in:
        for name in ["data_sets", "data_set_ncfg", "data_set_cfg_shift", "data_set_skip_cfg"]:
            fit_in.copy(name, f)
print("saved", itd_file)

#---- plot: single and double ratio versus nu, one line per pz (pz 0 is not drawn: all its points sit at nu = 0)
itd_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
itd_markers = ["o", "s", "^", "D", "v", "P", "X"]
panels = [("double ratio:  [M(p, w) / M(p, w_ref)] / [M_ref(w) / M_ref(w_ref)]", double_mean, double_err)]
if make_single:
    panels = [("single ratio:  M(p, w) / M_ref(w) x E_ref / E", single_mean, single_err)] + panels
fig, axes = plt.subplots(1, len(panels), figsize=(6.3 * len(panels), 5), squeeze=False)
for ax, (title, mean, err) in zip(axes[0], panels):
    mean = np.where(err < itd_plot_err_max, mean, np.nan)
    for i, pz in enumerate(itd_pz_list):
        if pz != 0:
            ax.errorbar(nu[:, i], mean[:, i], err[:, i], color=itd_colors[pz], marker=itd_markers[pz], ms=5, lw=0.8, elinewidth=1, capsize=2, label=f"pz = {pz}")
    ax.axhline(1, color="#52514e", lw=0.8, ls=":")
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(r"$\nu = P_z z$")
    ax.set_ylabel("Re")
    ax.set_title(title, fontsize=10)
axes[0, -1].legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
fig.suptitle(f"{hadron_tag}, forward, {n_jk_itd} cfg, HYP {hyp}: sGEVP, t0 {sgevp_t0}, {sgevp_tag}, reference p = (0,0,0)", fontsize=10)
fig.tight_layout()
itd_plot_file = os.path.join(save_dir, "sgevp_itd_plots", "forward", itd_name + ".png")
os.makedirs(os.path.dirname(itd_plot_file), exist_ok=True)
fig.savefig(itd_plot_file, dpi=150)
plt.show()
print("saved", itd_plot_file)

#---- the double ratio per pz next to the ratio fit, if its saved file has the same cfgs
if itd_compare_file is not None and os.path.exists(itd_compare_file):
    with h5py.File(itd_compare_file, "r") as f:
        same_cfg = np.array_equal(f["cfg_list"][()], itd_cfg_list)
        cmp_mean = f["double_ratio_mean"][()]       #(w, pz)
        cmp_err = f["double_ratio_err"][()]
        cmp_pz = list(f["pz_list"][()])
    if same_cfg:
        fig, axes = plt.subplots(2, 3, figsize=(14, 7.6), sharey=True)
        for ax, pz in zip(axes.ravel(), [pz for pz in itd_pz_list if pz != 0]):
            i = itd_pz_list.index(pz)
            ok = double_err[:, i] < itd_plot_err_max
            ax.errorbar(nu[ok, i], double_mean[ok, i], double_err[ok, i], color=itd_colors[pz], marker="o", ms=5, lw=0, elinewidth=1.1, capsize=2, label="sGEVP")
            if pz in cmp_pz:
                c = cmp_pz.index(pz)
                ok = cmp_err[:, c] < itd_plot_err_max
                ax.errorbar(nu[ok, i] + 0.02 * pz, cmp_mean[ok, c], cmp_err[ok, c], color="#52514e", marker="s", mfc="none", ms=5, lw=0, elinewidth=1.1, capsize=2, label="ratio fit")
            ax.axhline(1, color="#52514e", lw=0.8, ls=":")
            ax.grid(color="#e1e0d9", lw=0.6)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_title(f"pz = {pz}: {' '.join(sgevp_choice[pz])}", fontsize=10, loc="left")
            ax.set_xlabel(r"$\nu = P_z z$")
            ax.set_ylabel("double ratio")
            ax.tick_params(labelleft=True)
        axes[0, 0].legend(frameon=False, fontsize=9, loc="lower left")
        fig.suptitle(f"{hadron_tag}, forward, {n_jk_itd} cfg: double ratio from the sGEVP (t0 {sgevp_t0}) and from the saved ratio fit ({os.path.basename(itd_compare_file)})", fontsize=10)
        fig.tight_layout()
        cmp_plot_file = os.path.join(save_dir, "sgevp_itd_plots", "forward", itd_name + "_vs_ratio_fit.png")
        fig.savefig(cmp_plot_file, dpi=150)
        plt.show()
        print("saved", cmp_plot_file)
    else:
        print(f"{itd_compare_file} has other cfgs: no comparison")
else:
    print(f"no ratio-fit file to compare with at {itd_compare_file}")